# Step 1: Install the required packages/libraries 

In [ ]:
# Step 1: Install the required packages/libraries 

# pip install pandas
# pip install openpyxl

# Step 2: Import required libraries

In [4]:
import sqlite3
import pandas as pd
import os


# Step 3: Load/Import data & Build connection

In [10]:

# -------- Step 1: Load input file --------
# The input file is located inside:
# Task_1/input_tables/

file_path = r"input_tables/customer_sample_500.csv.xlsx"

# Read Excel file
df = pd.read_excel(file_path, dtype=str)

# Replace NaN values with empty string
df = df.fillna("")


# -------- Step 2: Create SQLite in-memory database --------
conn = sqlite3.connect(":memory:")


# -------- Step 3: Load DataFrame into SQLite table --------
df.to_sql("customers", conn, index=False, if_exists="replace")


# -------- Step 4: Create cursor for SQL queries --------
cursor = conn.cursor()


print("Data successfully loaded into SQLite table: customers")

Data successfully loaded into SQLite table: customers


# Step 4: Data Quality | Completeness Check

## Datasets using: 
- customer_sample_500


In [13]:
# ----- Completeness Check -----

def completeness_check(cursor, df, output_file="QualityCheck_results.xlsx"):
    """
    Perform a completeness data quality check on each column in the 'customers' table.

    This function checks for missing values in every column of the SQL table by
    identifying records where the column value is either:
        1. NULL
        2. An empty string ('') or whitespace

    The results are compiled into a DataFrame and written to an Excel file
    inside the 'output_files' folder.

    Parameters
    ----------
    cursor : sqlite3.Cursor
        Active database cursor used to execute SQL queries on the 'customers' table.

    df : pandas.DataFrame
        DataFrame containing the same columns as the SQL table. Used only to
        iterate through column names.

    output_file : str, optional
        Name of the Excel file where the completeness results will be stored.
        Default is "QualityCheck_results.xlsx".

    Returns
    -------
    pandas.DataFrame
        A DataFrame containing:
        - Column name
        - Number of missing values detected in that column

    Output
    ------
    The results are saved to:
        output_files/QualityCheck_results.xlsx

    The Excel file will contain a sheet named:
        "Completeness check"
    """

    # List to store completeness results for each column
    results = []

    # Loop through each column in the DataFrame
    # These column names correspond to columns in the SQL table
    for col in df.columns:

        # SQL query to count missing values
        # Missing values are defined as:
        #   - NULL values
        #   - Empty strings or whitespace
        query = f"""
        SELECT COUNT(*)
        FROM customers
        WHERE {col} IS NULL OR TRIM({col}) = ''
        """

        # Execute SQL query and retrieve the count of missing values
        missing = cursor.execute(query).fetchone()[0]

        # Store the result for the current column
        results.append({
            "Column": col,
            "Missing Values": missing
        })

    # Convert results list into a pandas DataFrame
    result_df = pd.DataFrame(results)

    # Define output folder path
    folder_path = "output_files"

    # Create the folder if it does not exist
    os.makedirs(folder_path, exist_ok=True)

    # Construct full path for the output Excel file
    file_path = os.path.join(folder_path, output_file)

    # Write results to Excel
    # If the file already exists, replace the sheet named "Completeness check"
    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:

        result_df.to_excel(writer, sheet_name="Completeness check", index=False)

    # Print confirmation message
    print("Output saved to", file_path)

    # Return the results DataFrame for further use if needed
    return result_df


# Run completeness check
result = completeness_check(cursor, df)

Output saved to output_files\QualityCheck_results.xlsx


# Step -5: Data Quality | Uniquenss check
## Datasets using:
- customer_sample_500

In [12]:
'''Uniqueness_check'''

import pandas as pd


def find_unique_columns_sql(cursor, df):
    """
    Identify columns in the SQL 'customers' table that contain only unique values.

    This function checks each column in the provided DataFrame and verifies
    whether the corresponding column in the SQL table contains unique values.
    A column is considered unique if the total row count equals the count
    of distinct values in that column.

    Parameters
    ----------
    cursor : database cursor
        Active database cursor used to execute SQL queries.

    df : pandas.DataFrame
        DataFrame containing the list of columns that exist in the SQL table.

    Returns
    -------
    list
        A list of column names that contain only unique values in the table.

    Notes
    -----
    This check is commonly used in data quality validation to detect
    candidate primary key columns or ensure uniqueness constraints.
    """

    # List to store columns that contain only unique values
    unique_columns = []

    # Loop through each column in the dataframe
    for col in df.columns:

        # SQL query to check if total row count equals distinct count
        query = f"""
        SELECT COUNT(*) = COUNT(DISTINCT {col})
        FROM customers
        """

        # Execute query and fetch result
        result = cursor.execute(query).fetchone()[0]

        # If result is True (1), column values are unique
        if result == 1:
            unique_columns.append(col)

    # Return list of unique columns
    return unique_columns


# -------- Step 1: Detect columns with unique values --------

# Run uniqueness detection function
unique_cols = find_unique_columns_sql(cursor, df)

# Convert list of unique columns into a DataFrame for reporting
unique_df = pd.DataFrame(unique_cols, columns=["Unique_Columns"])


# -------- Step 2: Check duplicate count for those columns --------

# List to store duplicate count results
duplicate_results = []

# Loop through columns that were detected as unique
for col in unique_cols:

    # SQL query to calculate number of duplicate values
    # Formula: total rows - distinct values
    query = f"""
    SELECT COUNT(*) - COUNT(DISTINCT {col})
    FROM customers
    """

    # Execute query and retrieve duplicate count
    duplicates = cursor.execute(query).fetchone()[0]

    # Append results to list as dictionary
    duplicate_results.append({
        "Column": col,
        "Duplicate_Count": duplicates
    })

# Convert duplicate results into a DataFrame
duplicate_df = pd.DataFrame(duplicate_results)


# -------- Step 3: Save results to Excel file --------

# Output Excel file path
file_path = "output_files/QualityCheck_results.xlsx"

# Write results into the same Excel file but different sheets
with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    # Sheet 1: Columns containing unique values
    unique_df.to_excel(writer, sheet_name="Unique Columns", index=False)

    # Sheet 2: Duplicate count for those columns
    duplicate_df.to_excel(writer, sheet_name="Duplicate Check", index=False)

# Confirmation message
print("Uniqueness results added to QualityCheck_results.xlsx in new sheets")

Uniqueness results added to QualityCheck_results.xlsx in new sheets


# Step-6 : Quality check | Validity check
## Datasets using :
- customer_sample_500 

In [14]:
# ------ Validity Check ------

def run_validity_checks(cursor):
    """
    Perform validity checks on specific columns in the 'customers' SQL table.

    This function executes a set of predefined validation rules to identify
    records that do not meet expected data formats or allowed values.

    The checks include:
        - Length validation for identifiers (SIRET, SIREN)
        - Country code length validation
        - Postal code format validation (digits only)
        - Allowed value checks for delivery/order block fields
        - Numeric validation for GERS values

    Each rule returns the count of records that violate the rule.

    Parameters
    ----------
    cursor : sqlite3.Cursor
        Active SQLite database cursor used to execute SQL queries.

    Output
    ------
    The validation results are written to the Excel file:
        output_files/QualityCheck_results.xlsx

    A sheet named "Validity_Check" will contain:
        - Check name
        - Number of invalid records detected
    """

    # Dictionary storing validation rule names and their SQL queries
    validity_rules = {

        # Check that SIRET numbers contain exactly 14 characters
        "Invalid_SIRET": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(SIRET) != 14
        """,

        # Check that SIREN numbers contain exactly 9 characters
        "Invalid_SIREN": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(SIREN) != 9
        """,

        # Country codes should contain exactly 2 characters
        "Invalid_Country_Code": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(COUNTRY_CD) != 2
        """,

        # SAP postal codes should contain only numeric characters
        "Invalid_SAP_Postal_Code": """
            SELECT COUNT(*) FROM customers
            WHERE SAP_POSTAL_ZIP_CD GLOB '*[^0-9]*'
        """,

        # LUCI postal codes should contain only numeric characters
        "Invalid_LUCI_Postal_Code": """
            SELECT COUNT(*) FROM customers
            WHERE LUCI_POSTAL_ZIP_CD GLOB '*[^0-9]*'
        """,

        # CUSTOMER_DELIVERY_BLOC must contain only allowed values
        "Invalid_Customer_Delivery_Block": """
            SELECT COUNT(*) FROM customers
            WHERE CUSTOMER_DELIVERY_BLOC NOT IN ('00','01')
        """,

        # GERS values should contain only numeric characters
        "Invalid_GERS": """
            SELECT COUNT(*) FROM customers
            WHERE GERS GLOB '*[^0-9]*'
        """
    }

    # Columns that should contain only allowed block values ('00' or '01')
    delivery_columns = [
        "CUSTOMER_ORDER_BLOCK",
        "CUSTOMER_SALES_DELIVERY_BLOC",
        "CUSTOMER_SALES_ORDER_BLOCK",
        "DELETION_BLOCK"
    ]

    # Dynamically create validation rules for the delivery block columns
    for col in delivery_columns:
        validity_rules[f"{col}_Invalid"] = f"""
            SELECT COUNT(*) FROM customers
            WHERE {col} NOT IN ('00','01')
        """

    # List to store validation results
    results = []

    # Execute each validation rule
    for rule, query in validity_rules.items():

        # Execute SQL query and retrieve invalid record count
        invalid_count = cursor.execute(query).fetchone()[0]

        # Store result in dictionary format
        results.append({
            "Check": rule,
            "Invalid_Count": invalid_count
        })

    # Convert results list into a pandas DataFrame
    validity_df = pd.DataFrame(results)

    # Existing Excel output file
    file_path = "output_files/QualityCheck_results.xlsx"

    # Append results to Excel in a new sheet named "Validity_Check"
    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        validity_df.to_excel(writer, sheet_name="Validity_Check", index=False)

    # Confirmation message
    print("Validity results added to QualityCheck_results.xlsx")


# Run validity checks
run_validity_checks(cursor)

Validity results added to QualityCheck_results.xlsx


# Step-7: Quality Checks | Consistency check
## Data sets using :
- customer_sample_500

In [15]:
# ------ Consistency Check -------

def run_consistency_checks(cursor):
    """
    Perform consistency validation checks on the 'customers' SQL table.

    This function evaluates predefined consistency rules to identify records
    where related columns contain conflicting or inconsistent values.

    Currently implemented rule:
        - Country_Mismatch:
          Checks whether the values in COUNTRY_CD and LUCI_COUNTRY_CD
          columns are different for the same record.

    Parameters
    ----------
    cursor : sqlite3.Cursor
        Active SQLite database cursor used to execute SQL queries.

    Output
    ------
    The results are written to the Excel file:
        output_files/QualityCheck_results.xlsx

    A new sheet named "Consistency_Check" will contain:
        - Check name
        - Number of mismatched records
        - Output file location
    """

    # Dictionary storing consistency rule names and corresponding SQL queries
    consistency_rules = {
        "Country_Mismatch": """
            SELECT COUNT(*)
            FROM customers
            WHERE COUNTRY_CD != LUCI_COUNTRY_CD
        """
    }

    # List to store results for each consistency rule
    results = []

    # Excel output file path
    file_path = "output_files/QualityCheck_results.xlsx"

    # Execute each consistency rule
    for rule, query in consistency_rules.items():

        # Run SQL query and retrieve mismatch count
        mismatch_count = cursor.execute(query).fetchone()[0]

        # Store the result in dictionary format
        results.append({
            "Check": rule,
            "Mismatch_Count": mismatch_count,
            "Output_Location": file_path
        })

    # Convert results list into a pandas DataFrame
    consistency_df = pd.DataFrame(results)

    # Write results to the Excel file in a new sheet
    # If the sheet already exists, it will be replaced
    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        consistency_df.to_excel(writer, sheet_name="Consistency_Check", index=False)

    # Confirmation message
    print(f"Consistency results saved at: {file_path}")


# Run consistency check
run_consistency_checks(cursor)

Consistency results saved at: output_files/QualityCheck_results.xlsx
